In [ ]:
# %%
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["CUDA_MODULE_LOADING"] = "LAZY"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices=false"
os.environ["TF_CUDNN_USE_AUTOTUNE"] = "0"

import numpy as np
import tensorflow as tf
import keras
from keras import layers, models, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

# ---- StateFarm paths ----
BASE_DIR = "/home/lurpd/DevelopmentWSL2/Datasets/statefarmdataset"
TRAIN_DIR = os.path.join(BASE_DIR, "imgs/train")
# TEST_DIR = os.path.join(BASE_DIR, "imgs/test")  # optional later

# ---- Training params ----
IMG_SIZE = 128
BATCH = 16          # start stable; bump to 32 later if it runs clean
SEED = 42
VAL_SPLIT = 0.2
WD = 1e-5
EPOCHS = 20


In [ ]:
# %%
# Data generators (simple + stable)
train_gen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VAL_SPLIT,
    rotation_range=5,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.10,
    brightness_range=[0.85, 1.15],
    horizontal_flip=True,
    fill_mode="nearest"
)

val_gen = ImageDataGenerator(rescale=1./255, validation_split=VAL_SPLIT)

train_generator = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode="rgb",
    batch_size=BATCH,
    class_mode="categorical",
    subset="training",
    seed=SEED,
    shuffle=True
)

val_generator = val_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode="rgb",
    batch_size=BATCH,
    class_mode="categorical",
    subset="validation",
    seed=SEED,
    shuffle=False
)

NUM_CLASSES = train_generator.num_classes
class_names = list(train_generator.class_indices.keys())
print("Classes:", train_generator.class_indices)
print("NUM_CLASSES:", NUM_CLASSES)


In [ ]:
# %%
# Simple CNN (scratch) with a stable head (GAP instead of Flatten)
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    layers.Conv2D(32, 3, padding="same", activation="relu",
                  kernel_regularizer=regularizers.l2(WD)),
    layers.MaxPooling2D(2),

    layers.Conv2D(64, 3, padding="same", activation="relu",
                  kernel_regularizer=regularizers.l2(WD)),
    layers.MaxPooling2D(2),

    layers.Conv2D(128, 3, padding="same", activation="relu",
                  kernel_regularizer=regularizers.l2(WD)),
    layers.MaxPooling2D(2),

    layers.Conv2D(256, 3, padding="same", activation="relu",
                  kernel_regularizer=regularizers.l2(WD)),

    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu",
                 kernel_regularizer=regularizers.l2(WD)),
    layers.Dropout(0.35),

    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.summary()


In [ ]:
# %%
# Warmup (forces PTX JIT for this exact shape early, reduces random mid-fit crashes)
x_warm = tf.random.uniform((BATCH, IMG_SIZE, IMG_SIZE, 3))
_ = model(x_warm, training=False)
print("Warmup OK")


In [ ]:
# %%
# Class weights (optional but usually helpful)
labels = np.unique(train_generator.classes)
weights = compute_class_weight(
    class_weight="balanced",
    classes=labels,
    y=train_generator.classes
)
class_weights = dict(enumerate(weights))
print("Class weights:", class_weights)


In [ ]:
# %%
# Compile + train
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=["accuracy"],
    jit_compile=False
)

callbacks = [
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint("statefarm_simple_best.keras", monitor="val_accuracy",
                    save_best_only=True, mode="max")
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights
)


In [ ]:
# %%
# Evaluate
val_generator.reset()
val_preds = model.predict(val_generator, verbose=0)
y_pred = np.argmax(val_preds, axis=1)
y_true = val_generator.classes

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(xticks_rotation=45)
plt.show()


In [ ]:
# %%
# Save final
model.save("statefarm_simple_final.keras")
print("Saved: statefarm_simple_final.keras")


In [ ]:
# %%
# Plot curves
tr_loss = history.history["loss"]
val_loss = history.history["val_loss"]
tr_acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
ep = range(1, len(tr_loss) + 1)

plt.plot(ep, tr_loss, label="Train")
plt.plot(ep, val_loss, label="Val")
plt.title("Loss")
plt.legend()
plt.show()

plt.plot(ep, tr_acc, label="Train")
plt.plot(ep, val_acc, label="Val")
plt.title("Accuracy")
plt.legend()
plt.show()
